In [1]:
from dataset import *
from evaluate import *
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
class NCImageDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []  # List to store tuples of (file_path, class_label)
        
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file == ".DS_Store":
                    continue
                file_path = os.path.join(root, file)
                folder_name = os.path.basename(root)
                
                self.samples.append(file_path)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        file_path = self.samples[idx]
        
        with rio.open(file_path) as src:
            image = src.read([1,2,3])  
            image = torch.tensor(image)  
        
        transform  = transforms.Compose([
                            transforms.ToPILImage(),
#                             transforms.RandomHorizontalFlip(),
#                             transforms.RandomRotation(degrees=30),
#                             transforms.CenterCrop(size=224),
                            transforms.ToTensor()
        ])
        sample = {'image': transform(image),
                 'file_path': file_path}
        return sample

In [3]:
nc_sites = '/work/srs108/crops/bridge/'
dataset = NCImageDataset(root_dir=nc_sites)

batch_size = 16

chowan_loader = DataLoader(dataset, batch_size=batch_size, shuffle = False)


In [4]:
model = CNN().to(device)
model.load_state_dict(torch.load('/work/srs108/planet_binary.pt'))


<All keys matched successfully>

In [5]:
def bloom_image_plotting(batch, output_binary):
    import matplotlib.pyplot as plt

    num_images = len(batch['image'])  # Number of images in the batch
    rows, cols = 4, 4  # 4x4 grid for a batch of 16 images

    # Create the figure and axes
    fig, axes = plt.subplots(rows, cols, figsize=(15, 15))  # Increase the figure size for larger images

    # Loop through the images and plot
    for i in range(num_images):
        row = i // cols  # Determine the row index
        col = i % cols   # Determine the column index
        axes[row, col].imshow(batch['image'][i].T)  # Transpose the image for correct orientation
        axes[row, col].set_title(f'Output: {output_binary[i].item()}')
        axes[row, col].axis('off')  # Hide axes

    # Turn off any unused axes if the number of images is less than 16
    for i in range(num_images, rows * cols):
        fig.delaxes(axes[i // cols, i % cols])

    # Tight layout to reduce spacing
    plt.tight_layout()
    plt.show()
    plt.close()


In [6]:
model.eval()
tracking = {'date': [], 'pred': []}
with torch.no_grad():
    for i, batch in tqdm(enumerate(chowan_loader)):
        x = batch['image'].float().to(device)
        p = batch['file_path'] 
        
        output = model(x)
#         logits = (output.cpu().detach().numpy()).astype(int)
        output_binary = (output.cpu().detach().numpy() >= 0).astype(int)
        

 
        torch.cuda.empty_cache()
        dates = [ datetime.datetime.strptime(fp.split('/')[-1][:8], "%Y%m%d").strftime("%m-%d-%Y") for fp in p]
        locs = [fp.split('/')[-3] for fp in p]
# #         bloom_image_plotting(batch, output_binary)
        tracking['date'].extend(dates) 
        tracking['pred'].extend(output_binary.reshape(-1).tolist())  
#         break

90it [04:22,  2.92s/it]


In [7]:
df = pd.DataFrame(tracking)
df.to_csv('bridge.csv', index=False)

In [6]:
df = pd.read_csv('csvs/alldata.csv', parse_dates=['date'],infer_datetime_format='%m/%d/%Y').sort_values(by='date')
df['date'] = df['date'].dt.strftime('%m-%d-%Y')

In [8]:
df = pd.read_csv('csvs/alldata_sort.csv', parse_dates=['date'],infer_datetime_format='%m/%d/%Y').sort_values(by='date')
df['date'] = df['date'].dt.strftime('%m-%d-%Y')

In [9]:
df1 = df.loc[df['spot'] == 'chowan_river']


In [14]:
df1.drop('spot', axis=1) 

,Unnamed: 0,date,label,pred
20,556,02-05-2017,0,0
23,633,02-12-2017,0,0
26,317,02-17-2017,0,0
40,337,03-04-2017,0,0
47,880,03-16-2017,0,0
...,...,...,...,...
7265,721,06-16-2024,0,0
7286,937,06-17-2024,0,0
7297,1117,06-25-2024,0,0
7296,15,06-25-2024,0,0


In [13]:
df1

,Unnamed: 0,date,spot,label,pred
20,556,02-05-2017,chowan_river,0,0
23,633,02-12-2017,chowan_river,0,0
26,317,02-17-2017,chowan_river,0,0
40,337,03-04-2017,chowan_river,0,0
47,880,03-16-2017,chowan_river,0,0
...,...,...,...,...,...
7265,721,06-16-2024,chowan_river,0,0
7286,937,06-17-2024,chowan_river,0,0
7297,1117,06-25-2024,chowan_river,0,0
7296,15,06-25-2024,chowan_river,0,0


In [7]:
# d = df['date']
# l = df['label']
# p = df['pred']
# fig, ax = plt.subplots(1,1, figsize = (20,8))

# ax.plot(d, l, c = 'r',alpha = 0.5)
# ax.plot(d,p, c = 'b',alpha = 0.5)
# ax.set_xticks(d[0::50])
# ax.set_xticklabels(d[0::50], rotation=45, ha='right')
# ax.set_title('')
# plt.show

In [8]:
df1 = df.loc[df['spot'] == 'chowan_river']
df2 = df.loc[df['spot'] == 'arrowhead_beach']
df3 = df.loc[df['spot'] == 'cannons_ferry']
df4 = df.loc[df['spot'] == 'point_comfort']
df5 = df.loc[df['spot'] == 'mt_gould']
df6 = df.loc[df['spot'] == 'rocky_hock_lane']


In [9]:
# df1 = df1.loc[(df1['label'] == 1) & (df1['pred'] ==1)].drop_duplicates(subset=['date'])
# df2 = df2.loc[(df2['label'] == 1) & (df2['pred'] ==1)].drop_duplicates(subset=['date'])
# df3 = df3.loc[(df3['label'] == 1) & (df3['pred'] ==1)].drop_duplicates(subset=['date'])
# df4 = df4.loc[(df4['label'] == 1) & (df4['pred'] ==1)].drop_duplicates(subset=['date'])
# df5 = df5.loc[(df5['label'] == 1) & (df5['pred'] ==1)].drop_duplicates(subset=['date'])
# df6 = df6.loc[(df6['label'] == 1) & (df6['pred'] ==1)].drop_duplicates(subset=['date'])


df1 = df1.loc[(df1['pred'] ==1)].drop_duplicates(subset=['date'])
df2 = df2.loc[(df2['pred'] ==1)].drop_duplicates(subset=['date'])
df3 = df3.loc[(df3['pred'] ==1)].drop_duplicates(subset=['date'])
df4 = df4.loc[(df4['pred'] ==1)].drop_duplicates(subset=['date'])
df5 = df5.loc[(df5['pred'] ==1)].drop_duplicates(subset=['date'])
df6 = df6.loc[(df6['pred'] ==1)].drop_duplicates(subset=['date'])

In [10]:
# Define sites for each DataFrame
sites = ['chowan_river', 'arrowhead_beach', 'cannons_ferry', 'point_comfort', 'mt_gould', 'rocky_hock_lane']
dfs = [df1, df2, df3, df4, df5, df6]

# Initialize an empty dictionary to store dates and corresponding site names
date_sites = {}

# Iterate through each DataFrame and its corresponding site name
for site_name, df in zip(sites, dfs):
    # Filter unique dates for the current site
    for date in df['date'].unique():
        # Add the date to the dictionary or append the site name if the date already exists
        if date not in date_sites:
            date_sites[date] = [site_name]
        else:
            date_sites[date].append(site_name)

# Sort the dictionary by date
sorted_date_sites = {date: sites for date, sites in sorted(date_sites.items())}

# Print or return the result
# print(sorted_date_sites)


In [11]:
from datetime import datetime

# Sort the dictionary by date (converted to datetime for proper sorting)
sorted_date_sites = {
    date: sites
    for date, sites in sorted(date_sites.items(), key=lambda x: datetime.strptime(x[0], '%m-%d-%Y'))
}

# Print or use the sorted dictionary
# print(sorted_date_sites)

In [12]:
dfs = [df1, df2, df3, df4, df5, df6]

# Combine all dates into one DataFrame
all_dates = pd.concat([df[['date']] for df in dfs])

# Count occurrences of each date
date_counts = all_dates['date'].value_counts().reset_index()

# # Rename columns to match desired output
date_counts.columns = ['date', 'count']

# Sort by date if necessary
# date_counts = date_counts.sort_values(by='date').reset_index(drop=True)
date_counts['date'] = pd.to_datetime(date_counts['date'])
date_counts['date']= date_counts['date'].dt.strftime('%Y-%m-%d')
dz = date_counts.sort_values(by='date')

In [1]:
d = dz['date']
l = dz['count']
# sns.set(style="darkgrid") 

fig, ax = plt.subplots(1,1, figsize = (15,8))

ax.scatter(d, l, c = 'k')
ax.set_xticks(d[0::3])
ax.set_xticklabels(d[0::3], rotation=90, ha='right')
ax.set_title('Counts of Model Predicted Blooms over Chowan River Sites', fontsize = 25)
ax.set_ylabel('Count of locations (6 total)')
ax.set_xlabel('Date')
plt.show
# fig.savefig('TotalCountFigure.png', transparent=False, facecolor='white', bbox_inches='tight')

In [34]:
dy = dz.loc[dz['count'] == 5]

In [15]:
dz.tail(15)

,date,count
89,2021-08-12,1
33,2021-09-11,2
87,2021-11-10,1
69,2022-05-21,1
86,2023-02-07,1
101,2023-02-08,1
63,2023-05-11,1
61,2023-07-11,1
11,2023-07-12,4
44,2023-07-13,2


In [43]:
# import matplotlib.pyplot as plt
# from matplotlib.ticker import MaxNLocator

# # Extract dates and sites
# dates = list(sorted_date_sites.keys())
# site_categories = list(set(site for sites in sorted_date_sites.values() for site in sites))  # Unique sites

# # Assign a numerical value to each site for plotting
# site_to_num = {site: i for i, site in enumerate(site_categories)}

# # Prepare data for the plot
# x = []
# y = []
# labels = []
# for date, sites in sorted_date_sites.items():
#     for site in sites:
#         x.append(date)
#         y.append(site_to_num[site])
#         labels.append(site)

# # Create the plot
# plt.figure(figsize=(18, 6))
# plt.scatter(x, y, color='b', alpha=0.7)

# # Add site labels
# # for i in range(len(labels)):
# #     plt.text(x[i], y[i] + 0.1, labels[i], fontsize=8, ha='center')

# # Format the axes
# plt.gca().xaxis.set_major_locator(MaxNLocator(nbins=15))  #
# plt.yticks(range(len(site_categories)), site_categories)
# plt.xticks(rotation=45)
# plt.title('Occurrence of Sites Over Time')
# plt.xlabel('Date')
# plt.ylabel('Sites')
# plt.tight_layout()
# plt.show()


In [2]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator

# Extract dates and sites
dates = list(sorted_date_sites.keys())
site_categories = list(set(site for sites in sorted_date_sites.values() for site in sites))  # Unique sites

# Assign a numerical value to each site for plotting
site_to_num = {site: i for i, site in enumerate(site_categories)}

# Prepare data for the plot
x = []
y = []
labels = []
colors = []

# Assign a color based on the number of sites per date
max_sites = 6  # Maximum number of sites
cmap = cm.get_cmap("tab10", max_sites)  # Generate a colormap
# Define custom ROYGBV colors
roygbv_colors = ["red", "orange", "yellow", "green", "blue", "violet"]

# Map number of sites to ROYGBV colors
num_sites_to_color = {n: roygbv_colors[n - 1] for n in range(1, max_sites + 1)}

for date, sites in sorted_date_sites.items():
    site_count = len(sites)
    color = num_sites_to_color.get(site_count, "gray")  # Use gray for unexpected site counts
    for site in sites:
        x.append(date)
        y.append(site_to_num[site])
        labels.append(site)
        colors.append(color)

# Create the plot
plt.figure(figsize=(18, 6))
scatter = plt.scatter(x, y, color=colors, alpha=0.7)
plt.gca().xaxis.set_major_locator(MaxNLocator(nbins=50))  #

# Format the axes
plt.yticks(range(len(site_categories)), site_categories)
plt.xticks(rotation=75)
plt.title('Occurrence of Sites Over Time (Color by Number of Sites Flagged)', fontsize = 25)
plt.xlabel('Date')
plt.ylabel('Sites')

# Add a legend
legend_labels = [f"{n} Site{'s' if n > 1 else ''}" for n in range(1, max_sites + 1)]
# Update legend_handles to use ROYGBV colors
legend_handles = [
    plt.Line2D([0], [0], marker='o', color=roygbv_colors[n - 1], linestyle='', markersize=10) 
    for n in range(1, max_sites + 1)
]
plt.legend(legend_handles, legend_labels, title="Sites Per Date", loc='upper right', bbox_to_anchor=(1.2, 1))

plt.tight_layout()
plt.savefig('SiteOccOverTime.png', transparent=False, facecolor='white', bbox_inches='tight')
plt.show()


In [33]:
sorted_date_sites['08-02-2017']

['chowan_river',
 'arrowhead_beach',
 'cannons_ferry',
 'point_comfort',
 'mt_gould',
 'rocky_hock_lane']

In [65]:
def bloom_tracking(date, root_dir, save = False):
    try: 
        dict_inst = sorted_date_sites[date]
    
        images_by_date = []

        for root, dirs, files in os.walk(root_dir):
            for file in sorted(files):
                if file == ".DS_Store":
                    continue
                file_date = str(file[4:6])+'-' + str(file[6:8]) +'-' + str(file[:4])
                if file_date == str(date) and root.split('/')[-1] == 'positive':
                    
                    image_path = os.path.join(root,file)
                    images_by_date.append(image_path)
                    break

        num_images = len(images_by_date)
        cols = min(3, num_images) 
        rows = (num_images + cols - 1) // cols 
        fig, ax = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))

        if num_images == 1:
            ax = [ax]

        
        elif rows > 1 or cols > 1:
            ax = ax.flatten()

        for i, img in enumerate(images_by_date):
            with rio.open(img) as src:
                image = src.read([1,2,3])
                
                image = np.transpose(image, (1,2,0))
                image = image.astype(float)
                
                image[...,1] *=1.5
                image = np.clip(image/ np.max(image), 0,1)
                ax[i].imshow(image) 
                ax[i].axis('off')  
                ax[i].set_title(img.split('/')[-3], fontsize=10)

        # Hide any unused axes
        for j in range(num_images, len(ax)):
            fig.delaxes(ax[j])

        fig.suptitle(f"Sites for {date}", fontsize=17, ha='center')  
        plt.tight_layout()
#         if save:  
#             fig.savefig('images_for_'+date + '.png', transparent=False, facecolor = 'white')

        plt.show()
    except Exception as e:
        print(e)
        print(f"Date {date} not in output")

In [3]:
bloom_tracking('08-02-2017', '/datacommons/carlsonlab/srs108/nc/', save = False)

In [4]:
for i in list(sorted_date_sites.keys()):
#     print(i)

    bloom_tracking(str(i), '/datacommons/carlsonlab/srs108/nc/', save = False)

In [5]:
newsite = '/datacommons/carlsonlab/srs108/a7/'
for fi in sorted(os.listdir(newsite)):
    with rio.open(os.path.join(newsite,fi)) as src:
        green = src.read([1])
        blue = src.read([2])
        unaltered = src.read([3])
        
#         print(fi)
        
#         image = np.transpose(green, (1,2,0))
#         image = image.astype(float)
#         print(image.shape)
#         image[...,1] *=1.5
#         image = np.clip(image/ np.max(image), 0,1)
        fig, ax = plt.subplots(1,3, figsize = (17,10))
        ax[0].imshow(np.moveaxis(green, 0,2)) 
        ax[0].axis('off')  
        ax[1].imshow(np.moveaxis(blue, 0,2)) 
        ax[1].axis('off')  
        ax[2].imshow(np.moveaxis(unaltered, 0,2))
        ax[2].axis('off')
        plt.show()
        break

In [6]:
newsite = '/datacommons/carlsonlab/srs108/crops/a7/'
for fi in sorted(os.listdir(newsite)):
    with rio.open(os.path.join(newsite,fi)) as src:
        green = src.read([1])
        blue = src.read([2])
        unaltered = src.read([1,2,3])
        print(fi)
        
#         image = np.transpose(green, (1,2,0))
#         image = image.astype(float)
#         print(image.shape)
#         image[...,1] *=1.5
#         image = np.clip(image/ np.max(image), 0,1)
        fig, ax = plt.subplots(1,3, figsize = (17,10))
        ax[0].imshow(np.moveaxis(green, 0,2)) 
        ax[0].axis('off')  
        ax[1].imshow(np.moveaxis(blue, 0,2)) 
        ax[1].axis('off')  
        ax[2].imshow(np.moveaxis(unaltered, 0,2))
        ax[2].axis('off')
        plt.show()